In [228]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

### DIRECTORY AND FILE PATH CONFIGURATION

In [204]:
ROOT_DIR=Path.cwd().parent
DATASET_DIR= ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw_loan_dataset.csv"
print(ROOT_DIR)
print(DATASET_DIR)
print(CSV_FILE_PATH)

/home/mdev/Documents/ml/ds-ml-bootcamp
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset/raw_loan_dataset.csv


### DATASET (EDA)

In [205]:
df = pd.read_csv(CSV_FILE_PATH)
df = df.rename(columns={'Approved':'Status'})
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [206]:
rows,columns = df.shape
print("ROWS:",rows)
print("COLUMNS:",columns)

ROWS: 103
COLUMNS: 7


In [207]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [208]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [209]:
def display_unique_values(col:str):
    print(f"\n{col}:",df[col].unique())

In [210]:
categorical_cols = ['Status','HasCollateral','PreviousDefaults']
for col in categorical_cols:
    display_unique_values(col)


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']


### CLEANING AND FORMATTING

In [211]:
def clean_numeric_col(col:str):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(r'[^0-9.]','',regex=True).astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('',np.nan)
    df[col] = pd.to_numeric(df[col],errors="coerce")

In [212]:
numeric_cols = ['Income','LoanAmount','CreditScore','EmploymentYears']
for col in numeric_cols:
    clean_numeric_col(col)
    print(df[col].head())


Cleaning column: Income
0    108810.0
1     96482.0
2     28478.0
3     25851.0
4     38396.0
Name: Income, dtype: float64

Cleaning column: LoanAmount
0    25800.0
1    11200.0
2    12100.0
3     7000.0
4    10700.0
Name: LoanAmount, dtype: float64

Cleaning column: CreditScore
0    537.0
1    524.0
2      NaN
3    561.0
4    527.0
Name: CreditScore, dtype: float64

Cleaning column: EmploymentYears
0     1.1
1    15.0
2     5.4
3    17.6
4     9.8
Name: EmploymentYears, dtype: float64


In [213]:
categorical_map = {
    '1':'Yes','Approved':'Yes','approved':'Yes','Y':'Yes','yse':'Yes',
    'Rejected':'No','rejected':'No','N':'No','0':'No'
    
}

def clean_format_categorical_col(col:str,categorical_map:dict):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(categorical_map).astype(str)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({'Nan':np.nan})

In [214]:
for col in categorical_cols:
    clean_format_categorical_col(col,categorical_map)
    display_unique_values(col)


Cleaning column: Status

Status: ['No' 'Yes']

Cleaning column: HasCollateral

HasCollateral: ['Yes' 'No' nan]

Cleaning column: PreviousDefaults

PreviousDefaults: ['No' nan 'Yes']


In [215]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

### FILL MISSING VALUES

In [216]:
def fill_with_median(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].median())
        
def fill_with_mode(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].mode()[0])

In [217]:
for col in numeric_cols:
    fill_with_median(col)
df.isna().sum()


Filling column: Income

Filling column: LoanAmount

Filling column: CreditScore

Filling column: EmploymentYears


Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [218]:
for col in categorical_cols:
    fill_with_mode(col)
    display_unique_values(col)
df.isna().sum()


Filling column: Status

Status: ['No' 'Yes']

Filling column: HasCollateral

HasCollateral: ['Yes' 'No']

Filling column: PreviousDefaults

PreviousDefaults: ['No' 'Yes']


Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Status              0
dtype: int64

### REMOVE DUPLICATES

In [219]:
before = df.shape
df = df.drop_duplicates()
print(f"Before dropping {before}(row,col) after dropped {df.shape}(row,col)")

Before dropping (103, 7)(row,col) after dropped (100, 7)(row,col)


### IQR capping on numeric columns

In [220]:
def iqr_fun(col:pd.Series,k=1.5):
    q1,q3 = col.quantile([0.25,0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

In [221]:
for col in numeric_cols:
    print(f"\nCapping column: {col}")
    lower,upper = iqr_fun(df[col])
    print("lower and upper before capping:(lower,upper)",(df[col].min(),df[col].max()))
    print("lower and upper of iqr:(lower,upper)",(lower,upper))
    df[col] = df[col].clip(lower=lower,upper=upper)
    print("lower and upper after capping:(lower,upper)",(df[col].min(),df[col].max()))


Capping column: Income
lower and upper before capping:(lower,upper) (25851.0, 250000.0)
lower and upper of iqr:(lower,upper) (-23827.875, 167619.125)
lower and upper after capping:(lower,upper) (25851.0, 167619.125)

Capping column: LoanAmount
lower and upper before capping:(lower,upper) (5000.0, 180000.0)
lower and upper of iqr:(lower,upper) (-16687.5, 66212.5)
lower and upper after capping:(lower,upper) (5000.0, 66212.5)

Capping column: CreditScore
lower and upper before capping:(lower,upper) (484.0, 920.0)
lower and upper of iqr:(lower,upper) (344.25, 962.25)
lower and upper after capping:(lower,upper) (484.0, 920.0)

Capping column: EmploymentYears
lower and upper before capping:(lower,upper) (0.5, 25.0)
lower and upper of iqr:(lower,upper) (-11.275, 35.525000000000006)
lower and upper after capping:(lower,upper) (0.5, 25.0)


### LABEL ENCODING YES/NO 1/0

In [222]:
for col in categorical_cols:
    print(f"\nEncoding column: {col}")
    df[col] = df[col].map({"Yes":1,"No":0}).astype(int)
    print(df[col].head())


Encoding column: Status
0    0
1    1
2    1
3    1
4    1
Name: Status, dtype: int64

Encoding column: HasCollateral
0    1
1    1
2    0
3    1
4    0
Name: HasCollateral, dtype: int64

Encoding column: PreviousDefaults
0    0
1    0
2    0
3    0
4    0
Name: PreviousDefaults, dtype: int64


In [223]:
df[categorical_cols].info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Status            100 non-null    int64
 1   HasCollateral     100 non-null    int64
 2   PreviousDefaults  100 non-null    int64
dtypes: int64(3)
memory usage: 3.1 KB


### CLASS BALANCE CHECK

In [227]:
print(df['Status'].value_counts())
class_ratio = df['Status'].value_counts(normalize=True).min()
print("Class ratio:",class_ratio)
if class_ratio < 0.3:
    print("\nWarning: severely imbalanced classes")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")



Status
1    66
0    34
Name: count, dtype: int64
Class ratio: 0.34

Class balance OK for baseline Accuracy (both classes well represented).


### SCALING WITH STANDARD SCALER

In [229]:
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df[numeric_cols].head()

,Income,LoanAmount,CreditScore,EmploymentYears
0,1.260277,0.019897,-1.191820,-1.656317
1,0.835658,-0.971972,-1.327610,0.336223
2,-1.506631,-0.910830,-0.136835,-1.039920
3,-1.597114,-1.257305,-0.941130,0.708929
4,-1.165022,-1.005941,-1.296274,-0.409187


In [230]:
df[categorical_cols].head()

,Status,HasCollateral,PreviousDefaults
0,0,1,0
1,1,1,0
2,1,0,0
3,1,1,0
4,1,0,0
